<a href="https://colab.research.google.com/github/ayse50/PyTorch/blob/main/casting_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Quality Control of Casting Products

##Problem
Manual inspection of casting products is time-consuming, prone to error, and leads to economic losses due to human inaccuracies.

##Solution
Development of a Deep Learning model for binary classification using PyTorch to automate and streamline the inspection process.

##Dataset
* Content: Grayscale images of pump impellers.

* Classes: 2 (Defective vs. OK).




## 1. Data Loading and Visualization

### Downloading Dataset from Kaggle

In [ ]:
!pip install opendatasets --upgrade --quiet

In [ ]:

import opendatasets as od
dataset_url = 'https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product'
od.download(dataset_url)


### Define paths to the existing datasets

In [ ]:
import os

base_path = '/content/real-life-industrial-dataset-of-casting-product/casting_data/casting_data'
train_dir = os.path.join(base_path, 'train')
test_dir = os.path.join(base_path, 'test')
train_ok_dir = os.path.join(train_dir, 'ok_front' )
train_def_dir = os.path.join(train_dir, 'def_front' )
test_ok_dir = os.path.join(test_dir, 'ok_front' )
test_def_dir = os.path.join(test_dir, 'def_front' )


### Visualizing class distribution across splits

In [ ]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Data distribution count
train_def = len(os.listdir(train_def_dir))
train_ok = len(os.listdir(train_ok_dir))
test_def = len(os.listdir(test_def_dir))
test_ok = len(os.listdir(test_ok_dir))

# Generating train and test DataFrames
daten = {'Teilung': ['Train', 'Train', 'Test', 'Test'],
         'Class': ['Ok', 'Defective', 'Ok', 'Defective'],
         'Anzahl': [train_ok, train_def, test_ok, test_def]}

df = pd.DataFrame(daten)

#Visualisierung der Aufteilung der Daten
sns.set_style('darkgrid')
plt.figure(figsize=(10, 7))
ax = sns.barplot(x='Teilung', y='Anzahl', hue='Class', data=df, palette="rocket")
plt.title('Dataset Distribution by Split and Class', fontsize=16)
plt.xlabel('Split', fontsize=12)
plt.ylabel('Number of images')

for container in ax.containers:
  ax.bar_label(container)

plt.tight_layout()

plt.savefig("data_distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
print(f'Training Set Ratio:{train_ok / train_def:.2f} (Ok to Defektive)')
print(f'Test Set Ratio:{test_ok / test_def:.2f} (Ok to Defektive)')

## 2. Data Preprocessing and Transformation

In [ ]:
import torch
import torchvision
from torchvision import transforms, datasets
from torchvision.transforms import ToTensor
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import numpy as np

SEED = 42
torch.manual_seed(SEED)

# Image transformation and normalization
transform_img_trn = transforms.Compose([transforms.Resize((224, 224)),
                                        transforms.Grayscale(num_output_channels=1),
                                        transforms.RandomRotation(15),
                                        transforms.RandomHorizontalFlip(),
                                        transforms.RandomVerticalFlip(),
                                        transforms.ColorJitter(brightness=0.3, contrast=0.3),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.5], std=[0.5])])

transform_img_tst = transforms.Compose([transforms.Resize((224, 224)),
                                        transforms.Grayscale(num_output_channels=1),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.5], std=[0.5])])

train_set = datasets.ImageFolder(root='/content/real-life-industrial-dataset-of-casting-product/casting_data/casting_data/train', transform=transform_img_trn)
test_set =datasets.ImageFolder(root='/content/real-life-industrial-dataset-of-casting-product/casting_data/casting_data/test', transform=transform_img_tst)

#Dataloader
BATCH_SIZE = 32

train_dataloader= DataLoader(train_set,
                             batch_size=BATCH_SIZE,
                             shuffle = True)


test_dataloader= DataLoader(test_set,
                             batch_size=BATCH_SIZE,
                             shuffle = False)


print(f"Dataloaders: {train_dataloader, test_dataloader}")
print(f"Length of train dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}")

In [ ]:
images, labels = next(iter(train_dataloader))
class_names = train_set.classes
images.shape, labels.shape

### Visualize 4 random samples from the training set
#### This serves as a sanity check to inspect the state of the images before model training.


In [ ]:

fig, axes = plt.subplots(2,2, figsize=(10,10))
random_idx = torch.randint(0, len(images), size=[4])
axes = axes.flatten()

for i, idx in enumerate(random_idx):
  img = images[idx]
  label = labels[idx].item()

  name = class_names[label]
  if 'ok' in name.lower():
    uberschrift = 'Ok'
  else:
    uberschrift = 'Defektive'

  axes[i].imshow(img.squeeze(), cmap=('gray'))
  axes[i].set_title(f'{uberschrift}, (index:{idx.item()})')
  axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Model, Loss Function, Optimizer, Training Loop, and Predictions for the Baseline Model

First, we establish a baseline model to serve as a benchmark for evaluating more complex architectures.

For the baseline, we use the simplest approach: A feed-forward network consisting of two nn.Linear() layers and one ReLU() layer in between.

In [ ]:
import torch
from torch import nn

# Setup device agnostic code
device = 'cuda' if torch.cuda.is_available() else'cpu'


flatten_model = nn.Flatten()
x = images[0]
output = flatten_model(x)

print(f'Form bevor flattening: {x.shape}')
print(f'Form nach flattening: {output.shape}')


### Baseline-Model (model_0)

In [ ]:

class ModelV0(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape:int):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=input_shape, out_features=hidden_units),
        nn.ReLU(),
        nn.Linear(in_features=hidden_units, out_features=output_shape)
    )

  def forward(self, x: torch.Tensor):
    return self.layer_stack(x)


# Setup model with Parameters
torch.manual_seed(42)

model_0 = ModelV0(input_shape= 50176,
                  hidden_units= 10,
                  output_shape= 2)
model_0.to(device)
print('Model setup on device:', {device})

next(model_0.parameters()).device

#### Setup loss, optimizer and evaluation metrics

In [ ]:
import requests
from pathlib import Path

# download Helper functions from Learn PyTorch repo
if Path('helper:functions.py').is_file():
  print('helper_functions.py already exists, skipping download...')
else:
  print('Downloading helper_functions.py')
  request = requests.get('https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py')
  with open('helper_functions.py','wb') as f: # wb= write binary, f=file
    f.write(request.content)


In [ ]:
# accuracy metric
from helper_functions import accuracy_fn

# loss function
loss_fn = nn.CrossEntropyLoss()

#optimizer
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.001)


In [ ]:
# function for timing the experiment
from timeit import default_timer as timer
def train_time(start: float, end:float, device: torch.device = None):
  total_time = end -start
  print(f'Train time on {device}: {total_time:.3f} seconds')
  return total_time


### Creating training/ testing loop & training Baseline-Model (with progress bar)

Functions for the loops

In [ ]:
#Function for Training
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
  train_loss, train_acc = 0, 0
  model.to(device)
  for batch, (X, y) in enumerate(data_loader):
    X, y = X.to(device), y.to(device)
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    train_loss += loss
    train_acc += accuracy_fn(y_true = y,
                             y_pred = y_pred.argmax(dim=1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  train_loss /= len(data_loader)
  train_acc /= len(data_loader)
  print(f'\nTrain loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%')

#Function for Test
def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device = device):
  test_loss, test_acc = 0, 0
  model.to(device)
  model.eval()
  with torch.inference_mode():
    for X, y in data_loader:
      X, y = X.to(device), y.to(device)
      test_pred = model(X)
      test_loss += loss_fn(test_pred, y)
      test_acc += accuracy_fn(y_true = y,
                              y_pred=test_pred.argmax(dim=1))

    test_loss /= len(data_loader)
    test_acc /= len(data_loader)
    print(f'\nTest loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%')


In [ ]:
from tqdm.auto import tqdm
torch.manual_seed(42)

from timeit import default_timer as timer
train_time_start = timer()
epochs = 5

for epoch in tqdm(range(epochs)):
  print(f'Epoch: {epoch}\n---------')
  train_step(model= model_0,
             data_loader= train_dataloader,
             loss_fn= loss_fn,
             optimizer= optimizer,
             accuracy_fn = accuracy_fn,
             device=device)
  test_step(model= model_0,
            data_loader= test_dataloader,
            loss_fn= loss_fn,
            accuracy_fn = accuracy_fn,
            device=device)

train_time_end = timer()
total_train_time_model_0 = train_time(start = train_time_start,
                                      end = train_time_end,
                                      device = device)


### Predictions and results for Baseline model

In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device: torch.device = device):


  loss , acc = 0, 0
  model.eval()
  with torch.inference_mode():
    for X, y in data_loader:
      X, y = X.to(device), y.to(device)
      y_pred = model(X)
      loss += loss_fn(y_pred, y)
      acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))

    loss /= len(data_loader)
    acc /= len(data_loader)

  return{'model_name': model.__class__.__name__,
           'model_loss': loss.item(),
           'model_acc': acc}



###Results of Baseline-Model

In [ ]:
model_0_result = eval_model(model_0, data_loader=test_dataloader,
                            loss_fn=loss_fn,
                            accuracy_fn=accuracy_fn,
                            device = device)

model_0_result

## 4. Model, Loss Funktion, Optimizer, training loop and predictions for model_1 (CNN)

###model_1 (CNN)

In [ ]:
class ModelV1(nn.Module):
  def __init__(self, input_shape:int, hidden_units:int, output_shape:int):
    super().__init__()

    # Block 1: Detect simple patterns like edges and textures
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels= input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )

    # Block 2: Combine simple patterns into complex shapes
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )

    # Classifier: Maps the extracted features to the final output classes
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*56*56,
                  out_features=output_shape)
    )

  def forward(self, x: torch.Tensor):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

torch.manual_seed(42)
model_1 = ModelV1(input_shape=1,
                  hidden_units=10,
                  output_shape=2).to(device)

model_1

###Setup loss and  optimizer for model_1

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_1.parameters(), lr=0.001)

### training and testing loop for model_1

In [ ]:
from tqdm.auto import tqdm
torch.manual_seed(42)

from timeit import default_timer as timer
train_time_start = timer()
epochs = 4

for epoch in tqdm(range(epochs)):
  print(f'Epoch: {epoch}\n---------')
  train_step(model= model_1,
             data_loader= train_dataloader,
             loss_fn= loss_fn,
             optimizer= optimizer,
             accuracy_fn = accuracy_fn,
             device=device)

  test_step(model= model_1,
            data_loader= test_dataloader,
            loss_fn= loss_fn,
            accuracy_fn = accuracy_fn,
            device=device)

train_time_end = timer()
total_train_time_model_1 = train_time(start = train_time_start,
                                      end = train_time_end,
                                      device = device)

###Prediction and results for model_1

In [ ]:
model_1_result = eval_model(model_1, data_loader=test_dataloader,
                            loss_fn=loss_fn,
                            accuracy_fn=accuracy_fn,
                            device = device)

model_1_result

## 5. Model, Loss Funktion, Optimizer, training loop and predictions for model_2

###model_2 (ResNet18)

In [ ]:
from torchvision import models
from torch import nn

model_2 = models.resnet18(weights= 'DEFAULT')
model_2.conv1 = nn.Conv2d(1, 64, kernel_size = 7, stride=2, padding=3, bias=False)
in_features =model_2.fc.in_features
model_2.fc = nn.Linear(in_features, 2)
model_2= model_2.to(device)

###Setup loss and  optimizer for model_2

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model_2.parameters(), lr=0.0001)

### training and testing loop for model_2

In [ ]:
from tqdm.auto import tqdm
torch.manual_seed(42)

from timeit import default_timer as timer
train_time_start = timer()
epochs = 4

for epoch in tqdm(range(epochs)):
  print(f'Epoch: {epoch}\n---------')
  train_step(model= model_2,
             data_loader= train_dataloader,
             loss_fn= loss_fn,
             optimizer= optimizer,
             accuracy_fn = accuracy_fn,
             device=device)

  test_step(model= model_2,
            data_loader= test_dataloader,
            loss_fn= loss_fn,
            accuracy_fn = accuracy_fn,
            device=device)

train_time_end = timer()
total_train_time_model_2 = train_time(start = train_time_start,
                                      end = train_time_end,
                                      device = device)

###Prediction and results for model_2

In [ ]:
model_2_result = eval_model(model_2, data_loader=test_dataloader,
                            loss_fn=loss_fn,
                            accuracy_fn=accuracy_fn,
                            device = device)

model_2_result

## Compare model results and training time

In [ ]:
import pandas as pd
compare = pd.DataFrame([model_0_result, model_1_result, model_2_result])
compare['training_time'] = [total_train_time_model_0, total_train_time_model_1, total_train_time_model_2]
compare

# 6. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch

model_list = [model_0, model_1, model_2]
model_name = ['ModelV0 (Baseline)', 'ModelV1 (CNN)', 'ModelV2 (ResNet18)']
fig, axes = plt.subplots(1,3, figsize=(18, 5))

for i, model in enumerate(model_list):
  y_preds = []
  y_true = []

  model.eval()
  with torch.inference_mode():
    for X, y in test_dataloader:
      X, y = X.to(device), y.to(device)
      logits = model(X)
      preds = torch.argmax(logits, dim=1)

      y_preds.extend(preds.cpu().numpy())
      y_true.extend(y.cpu().numpy())


  CM = confusion_matrix(y_true, y_preds)

  disp = ConfusionMatrixDisplay(confusion_matrix=CM, display_labels=class_names)
  disp.plot(ax = axes[i], cmap='Blues', colorbar=False)
  axes[i].set_title(model_name[i])
  axes[i].grid(False)


plt.tight_layout()
plt.savefig('confusion_matrix_casting_product.png', dpi=300)
plt.show()
